# Phishing Detection with Naive Bayes
#### Author: Ruben Huerta
#### Course: Advanced Data Analytics
##### Student ID: rah107

In [1]:
# General dataframe imports
import pandas as pd
import numpy as np

# sklearn imports
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn import naive_bayes
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import text
import sklearn.feature_extraction
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

# Stopwords
import nltk

# Sentiment Analysis
from textblob import TextBlob

# Regular Expressions
import re

# Preprocessing
import gensim

Dataset Information

`CEAS_08.csv`: Contains CEAS Phishing Email Data. The features include sender, receiver, date, subject, body, urls.

`Enron.csv`: Contains the Enron dataset. The features include subject, body.

`Ling.csv`: Contains Ling Phishing Email Dataset. The features include subject, body.

`Nazario.csv`: Contains Nazario Phishing Email Dataset. The features include sender, receiver, date, subject, body, urls.

`Nigerian_Fraud.csv`: Contains the Nigerian Fraud Phishing Email Dataset. The features include sender, receiver, date, subject, body, urls.

`Spam_Assassin.csv`: Contains the Spam Assasin Phishing Email Dataset. The features include sender, receiver, date, subject, body, urls.

`phishing_email.csv`: Combination of all the six datasets. The "text_combined" column is the central element of the final dataset in this phishing email analysis. It combines the subject line, the body, date, and sender email text of the emails from the initial datasets.

In [2]:
p_df = pd.read_csv('dataset/combined_phishing_emails.csv')

In [3]:
p_df.head(4)

,subject,body,label,source_file
0,Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,dataset/CEAS_08.csv
1,Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,dataset/CEAS_08.csv
2,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,dataset/CEAS_08.csv
3,Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,dataset/CEAS_08.csv


In [4]:
import json, re
import pandas as pd
from nrclex import NRCLex

with open("phishing_keywords.json") as f:
    lexicon = json.load(f)
lexicon = {k: v for k, v in lexicon.items() if k != "_meta"}

nrc = NRCLex()

def get_emo(t):
    tokens = re.findall(r"[a-z']+", t) if isinstance(t, str) else []
    nrc.load_token_list(tokens)
    return nrc.affect_frequencies

def extract_features(df, text_col, prefix):
    df[f"{text_col}_lower"] = df[text_col].fillna("").astype(str).str.lower()
    return df

p_df = extract_features(p_df, text_col="subject", prefix="subject")
p_df = extract_features(p_df, text_col="body", prefix="body")

In [5]:
p_df.head(3)

,subject,body,label,source_file,subject_lower,body_lower
0,Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,dataset/CEAS_08.csv,never agree to be a loser,"buck up, your troubles caused by small dimensi..."
1,Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,dataset/CEAS_08.csv,befriend jenna jameson,\nupgrade your sex and pleasures with these te...
2,CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,dataset/CEAS_08.csv,cnn.com daily top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...


In [6]:
import html

EMAIL_PATTERN = re.compile(r"\b[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}\b")
URL_PATTERN = re.compile(r"""(?xi)\b(?<!@)(?:(?:https?|hxxps?):\/\/|[a-z]+?\.)[a-z0-9\-._~%]+(?:\.[a-z]{2,})(?:\/[^\s"'<>]*)?""")
SEP_PATTERN = re.compile(r"[>=+\-~_]{4,}")
HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
WHITESPACE_PATTERN = re.compile(r"\s+")

for col in ["subject_lower", "body_lower"]:
    p_df[col] = p_df[col].apply(html.unescape)                       
    p_df[col] = p_df[col].apply(lambda t: HTML_TAG_PATTERN.sub(" ", t)) 
    p_df[col] = p_df[col].apply(lambda t: EMAIL_PATTERN.sub(" ", t))
    p_df[col] = p_df[col].apply(lambda t: URL_PATTERN.sub(" ", t))
    p_df[col] = p_df[col].apply(lambda t: SEP_PATTERN.sub(" ", t))
    p_df[col] = p_df[col].apply(lambda t: WHITESPACE_PATTERN.sub(" ", t).strip())

In [7]:
p_df.head(3)
p_df.to_csv("dataset/combined_phishing_emails_processed.csv", index=False)

In [8]:
# split first, before fitting any vectorizer
train_idx, test_idx = train_test_split(p_df.index, test_size=0.33, random_state=42)
 
X_train_df = p_df.loc[train_idx]
X_test_df = p_df.loc[test_idx]
y_train = p_df.loc[train_idx, "label"]
y_test = p_df.loc[test_idx, "label"]
 
subject_vec = TfidfVectorizer(min_df=3, max_df=0.9)
body_vec = TfidfVectorizer(min_df=3, max_df=0.9)
 
X_train_subject = subject_vec.fit_transform(X_train_df["subject_lower"])
X_train_body = body_vec.fit_transform(X_train_df["body_lower"])
X_test_subject = subject_vec.transform(X_test_df["subject_lower"])
X_test_body = body_vec.transform(X_test_df["body_lower"])
 
# TF-IDF only -- no numeric features stacked in this time
X_train = hstack([X_train_subject, X_train_body])
X_test = hstack([X_test_subject, X_test_body])
 
print("Subject vocab size:", len(subject_vec.get_feature_names_out()))
print("Body vocab size:", len(body_vec.get_feature_names_out()))
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
# Initialize a multinomial Naive Bayes model
model_nb = naive_bayes.MultinomialNB()

# Fit/train the model using the training data
model_nb.fit(X_train, y_train)

# Use the model to make prediction using the testing data
y_pred = model_nb.predict(X_test)

print("\n", classification_report(y_test, y_pred))

Subject vocab size: 8793
Body vocab size: 81619
X_train shape: (55265, 90412)
X_test shape: (27221, 90412)

               precision    recall  f1-score   support

           0       0.97      0.98      0.97     13099
           1       0.98      0.97      0.97     14122

    accuracy                           0.97     27221
   macro avg       0.97      0.97      0.97     27221
weighted avg       0.97      0.97      0.97     27221



In [9]:
# Initialize a logistic regression model
model_lr = LogisticRegression(random_state=42)

# Fit/train the model using the training data
model_lr.fit(X_train, y_train)

# use the model to make prediction using the testing data
y_pred_lr = model_lr.predict(X_test)

print("\n", classification_report(y_test, y_pred_lr))


               precision    recall  f1-score   support

           0       0.99      0.98      0.98     13099
           1       0.98      0.99      0.98     14122

    accuracy                           0.98     27221
   macro avg       0.98      0.98      0.98     27221
weighted avg       0.98      0.98      0.98     27221



In [10]:
def preprocess_new_data(df, lexicon, nrc):
    df = extract_features(df, text_col="subject", prefix="subject")
    df = extract_features(df, text_col="body", prefix="body")

    for col in ["subject_lower", "body_lower"]:
        df[col] = df[col].apply(html.unescape)
        df[col] = df[col].apply(lambda t: HTML_TAG_PATTERN.sub(" ", t))
        df[col] = df[col].apply(lambda t: EMAIL_PATTERN.sub(" ", t))
        df[col] = df[col].apply(lambda t: URL_PATTERN.sub(" ", t))
        df[col] = df[col].apply(lambda t: SEP_PATTERN.sub(" ", t))
        df[col] = df[col].apply(lambda t: WHITESPACE_PATTERN.sub(" ", t).strip())

    return df

In [11]:
def split_subject_body(text):
    text = str(text)
    if text.startswith("Subject:"):
        first_line, _, rest = text.partition("\n")
        subject = first_line.replace("Subject:", "").strip()
        body = rest.strip()
    else:
        subject, body = "", text
    return subject, body

new_df = pd.read_csv("dataset/phishing_legit_dataset_KD_10000.csv")
new_df[["subject", "body"]] = new_df["text"].apply(lambda t: pd.Series(split_subject_body(t)))

new_df = preprocess_new_data(new_df, lexicon, nrc)

X_new_subject = subject_vec.transform(new_df["subject_lower"])
X_new_body = body_vec.transform(new_df["body_lower"])
X_new = hstack([X_new_subject, X_new_body])
y_true = new_df["label"]

In [12]:
results = {}
models = {
    "Naive Bayes": model_nb,
    "Logistic Regression": model_lr
}

for name, m in models.items():
    y_pred = m.predict(X_new)
    print(f"=== {name} ===")
    print(classification_report(y_true, y_pred))
    results[name] = classification_report(y_true, y_pred, output_dict=True)["weighted avg"]

pd.DataFrame(results).T

=== Naive Bayes ===
              precision    recall  f1-score   support

           0       0.58      1.00      0.74      4000
           1       1.00      0.53      0.69      6000

    accuracy                           0.72     10000
   macro avg       0.79      0.76      0.71     10000
weighted avg       0.83      0.72      0.71     10000

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.88      1.00      0.93      4000
           1       1.00      0.91      0.95      6000

    accuracy                           0.94     10000
   macro avg       0.94      0.95      0.94     10000
weighted avg       0.95      0.94      0.94     10000



,precision,recall,f1-score,support
Naive Bayes,0.833166,0.7159,0.709149,10000.0
Logistic Regression,0.948816,0.9424,0.942863,10000.0
